# 🧹 Carlist.my Data Cleaning Pipeline
---
**Source:** BeautifulSoup scrape of carlist.my (100,005 rows × 11 columns)  
**Goal:** Produce a clean, analysis-ready `carlist_cleaned.csv`

### Column inventory
| Column | Raw Format | Issue |
|---|---|---|
| `price` | `RM 63,999` | String → numeric |
| `monthly_payment` | `RM 830 / month` | String → numeric, 209 nulls |
| `mileage` | `75 - 80K KM` | Range string → midpoint numeric |
| `seller_type` | Noisy long text | Extract clean label |
| `location` | `State , City` | Split into two columns |
| `year` | int | Filter outliers |


In [4]:
# ── 1. Imports & Setup ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import time
import tracemalloc
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.0f}'.format)

print("✅ Imports OK")
print(f"Pandas {pd.__version__} | NumPy {np.__version__}")


✅ Imports OK
Pandas 2.2.2 | NumPy 2.0.2


In [5]:
# Step 1: Run this cell to upload your file
from google.colab import files
uploaded = files.upload()  # A file picker will appear — select carlist_100k.csv

Saving carlist_100k_pw.csv to carlist_100k_pw.csv


In [6]:
# ── 2. Load Raw Data ────────────────────────────────────────────────────────
tracemalloc.start()
t0 = time.perf_counter()

RAW_PATH = 'carlist_100k_pw.csv'
df_raw = pd.read_csv(RAW_PATH)

load_time = time.perf_counter() - t0
mem_current, mem_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"✅ Loaded in {load_time:.3f}s")
print(f"   Shape      : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"   Memory used: {mem_peak / 1e6:.1f} MB (peak)")
print()
print(df_raw.dtypes)


✅ Loaded in 1.123s
   Shape      : 150,015 rows × 11 columns
   Memory used: 135.4 MB (peak)

id                 float64
title               object
year                 int64
price               object
monthly_payment     object
mileage             object
transmission        object
seller_type         object
location            object
condition           object
url                 object
dtype: object


In [7]:
# ── 3. Baseline Quality Report ──────────────────────────────────────────────
print("=" * 55)
print("BASELINE QUALITY REPORT")
print("=" * 55)
print(f"Total rows          : {len(df_raw):,}")
print(f"Duplicate rows      : {df_raw.duplicated().sum():,}")
print()
print("Missing values per column:")
miss = df_raw.isnull().sum()
for col, n in miss.items():
    pct = n / len(df_raw) * 100
    bar = '█' * int(pct) + '░' * (10 - int(pct))
    print(f"  {col:<18} {n:>5} ({pct:5.1f}%) [{bar}]")
print()
print("Year range:", df_raw['year'].min(), "–", df_raw['year'].max())
print("Condition  :", df_raw['condition'].value_counts().to_dict())
print("Transmission:", df_raw['transmission'].value_counts().to_dict())


BASELINE QUALITY REPORT
Total rows          : 150,015
Duplicate rows      : 0

Missing values per column:
  id                     1 (  0.0%) [░░░░░░░░░░]
  title                  0 (  0.0%) [░░░░░░░░░░]
  year                   0 (  0.0%) [░░░░░░░░░░]
  price                  0 (  0.0%) [░░░░░░░░░░]
  monthly_payment      187 (  0.1%) [░░░░░░░░░░]
  mileage                0 (  0.0%) [░░░░░░░░░░]
  transmission           0 (  0.0%) [░░░░░░░░░░]
  seller_type           55 (  0.0%) [░░░░░░░░░░]
  location               0 (  0.0%) [░░░░░░░░░░]
  condition              0 (  0.0%) [░░░░░░░░░░]
  url                    0 (  0.0%) [░░░░░░░░░░]

Year range: 1962 – 2026
Condition  : {'USED': 90695, 'RECON': 56785, 'NEW': 2535}
Transmission: {'Automatic': 142285, 'Manual': 7730}


---
## 🔧 Cleaning Steps
### Step 1 – Remove duplicates

In [8]:
# ── 4. Remove Duplicates ────────────────────────────────────────────────────
df = df_raw.copy()
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed : {before - len(df):,}")
print(f"Rows remaining     : {len(df):,}")


Duplicates removed : 0
Rows remaining     : 150,015


### Step 2 – Clean `price` → numeric (MYR)

In [9]:
# ── 5. Price Cleaning ───────────────────────────────────────────────────────
def parse_price(val):
    """'RM 63,999' → 63999.0"""
    if pd.isna(val):
        return np.nan
    cleaned = re.sub(r'[^\d.]', '', str(val))
    return float(cleaned) if cleaned else np.nan

df['price_myr'] = df['price'].apply(parse_price)

# Sanity-filter: keep prices between RM 1,000 and RM 5,000,000
mask_price = df['price_myr'].between(1_000, 5_000_000)
print(f"Price outliers removed : {(~mask_price).sum():,}")
df = df[mask_price].copy()

print(f"price_myr range : RM {df['price_myr'].min():,.0f} – RM {df['price_myr'].max():,.0f}")
print(f"price_myr median: RM {df['price_myr'].median():,.0f}")


Price outliers removed : 112
price_myr range : RM 1,288 – RM 4,999,999
price_myr median: RM 140,000


### Step 3 – Clean `monthly_payment` → numeric

In [10]:
# ── 6. Monthly Payment Cleaning ─────────────────────────────────────────────
def parse_monthly(val):
    """'RM 830 / month' → 830.0"""
    if pd.isna(val):
        return np.nan
    # If already numeric, return as-is
    if isinstance(val, (int, float)):
        return float(val)
    nums = re.findall(r'[\d,]+', str(val))
    if not nums:
        return np.nan
    return float(nums[0].replace(',', ''))

df['monthly_payment_myr'] = df['monthly_payment'].apply(parse_monthly)

# Fill nulls with median
median_mp = df['monthly_payment_myr'].median()
df['monthly_payment_myr'].fillna(median_mp, inplace=True)

print(f"monthly_payment nulls filled with median: RM {median_mp:,.0f}")
print(f"monthly_payment_myr range: RM {df['monthly_payment_myr'].min():,.0f} – RM {df['monthly_payment_myr'].max():,.0f}")

monthly_payment nulls filled with median: RM 1,815
monthly_payment_myr range: RM 1 – RM 99,800


### Step 4 – Clean `mileage` → midpoint numeric (km)

In [11]:
# ── 7. Mileage Cleaning ─────────────────────────────────────────────────────
def parse_mileage(val):
    """
    '75 - 80K KM'  → 77500.0   (midpoint)
    '120 - 125K KM'→ 122500.0
    Handles both 'K KM' and plain numbers.
    """
    if pd.isna(val):
        return np.nan
    val = str(val).upper().replace(',', '')
    nums = re.findall(r'[\d.]+', val)
    if not nums:
        return np.nan
    multiplier = 1000 if 'K' in val else 1
    values = [float(n) * multiplier for n in nums[:2]]
    return sum(values) / len(values)

df['mileage_km'] = df['mileage'].apply(parse_mileage)

print(f"mileage_km range : {df['mileage_km'].min():,.0f} – {df['mileage_km'].max():,.0f} km")
print(f"mileage_km median: {df['mileage_km'].median():,.0f} km")
print(f"Null mileage_km  : {df['mileage_km'].isnull().sum()}")


mileage_km range : 1,000 – 2,732,023,000 km
mileage_km median: 47,500 km
Null mileage_km  : 137


### Step 5 – Clean `seller_type` (extract label only)

In [12]:
# ── 8. Seller Type Cleaning ─────────────────────────────────────────────────
KNOWN = ['Dealer', 'Sales Agent', 'Private', 'Broker']

def parse_seller(val):
    """Extract first matching label; unknown → 'Other'"""
    if pd.isna(val):
        return 'Unknown'
    for label in KNOWN:
        if label.lower() in str(val).lower():
            return label
    return 'Other'

df['seller_type_clean'] = df['seller_type'].apply(parse_seller)

print("Seller type distribution:")
print(df['seller_type_clean'].value_counts().to_string())


Seller type distribution:
seller_type_clean
Dealer         76047
Sales Agent    72034
Broker          1236
Private          531
Unknown           55


### Step 6 – Split `location` → `state` + `city`

In [13]:
# ── 9. Location Splitting ────────────────────────────────────────────────────
# Format: 'State , City'  or  'State'
location_split = df['location'].str.split(r'\s*,\s*', n=1, expand=True)
df['state'] = location_split[0].str.strip()
df['city']  = location_split[1].str.strip() if 1 in location_split.columns else ''
df['city'].fillna('', inplace=True)

print("Top 10 states:")
print(df['state'].value_counts().head(10).to_string())


Top 10 states:
state
Selangor           56908
Kuala Lumpur       47922
Johor              32409
Perak               4300
Penang              3551
Melaka              1138
Pahang              1124
Negeri Sembilan      873
Kedah                636
Sabah                425


### Step 7 – Filter unrealistic `year` values

In [14]:
# ── 10. Year Filtering ───────────────────────────────────────────────────────
CURRENT_YEAR = 2026
before = len(df)
df = df[(df['year'] >= 1980) & (df['year'] <= CURRENT_YEAR)].copy()
print(f"Rows removed (year outliers): {before - len(df):,}")
print(f"Year range after filter: {df['year'].min()} – {df['year'].max()}")


Rows removed (year outliers): 21
Year range after filter: 1980 – 2026


### Step 8 – Standardise `condition` & `transmission` (Title Case)

In [15]:
# ── 11. Standardise Categorical Columns ─────────────────────────────────────
df['condition']    = df['condition'].str.strip().str.title()
df['transmission'] = df['transmission'].str.strip().str.title()

print("condition    :", df['condition'].value_counts().to_dict())
print("transmission :", df['transmission'].value_counts().to_dict())


condition    : {'Used': 90603, 'Recon': 56746, 'New': 2533}
transmission : {'Automatic': 142177, 'Manual': 7705}


### Step 9 – Drop redundant raw columns & finalise schema

In [16]:
# ── 12. Finalise Schema ─────────────────────────────────────────────────────
KEEP_COLS = [
    'id', 'title', 'year',
    'price_myr', 'monthly_payment_myr',
    'mileage_km', 'transmission',
    'seller_type_clean', 'state', 'city',
    'condition', 'url'
]

df_clean = df[KEEP_COLS].rename(columns={'seller_type_clean': 'seller_type'})
df_clean = df_clean.reset_index(drop=True)

print(f"✅ Final shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print()
print(df_clean.dtypes)
print()
print(df_clean.head(3).to_string())


✅ Final shape : 149,882 rows × 12 columns

id                     float64
title                   object
year                     int64
price_myr              float64
monthly_payment_myr    float64
mileage_km             float64
transmission            object
seller_type             object
state                   object
city                    object
condition               object
url                     object
dtype: object

          id                                                                          title  year  price_myr  monthly_payment_myr  mileage_km transmission  seller_type     state    city condition                                                                                                                    url
0 18,861,588   2025 Honda Civic 1.5 RS Sedan - HIGH SPEC - SERVICE RECORD WARRANTY BODY KIT  2025    119,888                1,554      12,500    Automatic  Sales Agent  Selangor  Cheras      Used  https://www.carlist.my/used-cars/2025-honda-civic-1-5-rs-h

In [17]:
# ── 13. Final Quality Report ─────────────────────────────────────────────────
print("=" * 55)
print("FINAL QUALITY REPORT")
print("=" * 55)
print(f"Total rows remaining : {len(df_clean):,}")
print(f"Rows cleaned away    : {len(df_raw) - len(df_clean):,}")
print()
print("Missing values per column:")
miss_clean = df_clean.isnull().sum()
for col, n in miss_clean.items():
    pct = n / len(df_clean) * 100
    flag = '⚠️' if n > 0 else '✅'
    print(f"  {flag} {col:<22} {n:>5} ({pct:.2f}%)")
print()
print("Numeric summary:")
print(df_clean[['price_myr', 'monthly_payment_myr', 'mileage_km', 'year']].describe().applymap('{:,.1f}'.format))


FINAL QUALITY REPORT
Total rows remaining : 149,882
Rows cleaned away    : 133

Missing values per column:
  ⚠️ id                         1 (0.00%)
  ✅ title                      0 (0.00%)
  ✅ year                       0 (0.00%)
  ✅ price_myr                  0 (0.00%)
  ✅ monthly_payment_myr        0 (0.00%)
  ⚠️ mileage_km               137 (0.09%)
  ✅ transmission               0 (0.00%)
  ✅ seller_type                0 (0.00%)
  ✅ state                      0 (0.00%)
  ✅ city                       0 (0.00%)
  ✅ condition                  0 (0.00%)
  ✅ url                        0 (0.00%)

Numeric summary:
         price_myr monthly_payment_myr       mileage_km       year
count    149,882.0           149,882.0        149,745.0  149,882.0
mean     242,172.2             3,129.6      1,195,019.0    2,019.8
std      368,191.7             4,743.3     14,244,500.5        4.1
min        1,288.0                 1.0          1,000.0    1,980.0
25%       56,800.0               736.0        

In [18]:
# ── 14. Save Cleaned Data ────────────────────────────────────────────────────
import os

OUT_PATH = '/content/carlist_cleaned.csv'
df_clean.to_csv(OUT_PATH, index=False)

size_mb = os.path.getsize(OUT_PATH) / 1e6
print(f"✅ Saved: {OUT_PATH}")
print(f"   File size: {size_mb:.1f} MB")
print(f"   Rows     : {len(df_clean):,}")
print(f"   Columns  : {df_clean.shape[1]}")

# Download to your local machine
from google.colab import files
files.download(OUT_PATH)

✅ Saved: /content/carlist_cleaned.csv
   File size: 39.3 MB
   Rows     : 149,882
   Columns  : 12


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>